In [1]:
import pandas as pd
import numpy as np
import os
from scipy import integrate
from scipy.stats import entropy, skew, kurtosis

In [ ]:
# All data (to be later filtered as we need)

column_names = [
    'Left_Hallux_raw', 'Right_Hallux_raw',
    'Left_Toes_raw', 'Right_Toes_raw',
    'Left_Met1_raw', 'Left_Met3_raw', 'Left_Met5_raw',
    'Right_Met1_raw', 'Right_Met3_raw', 'Right_Met5_raw', 
    'Left_Arch_raw', 'Right_Arch_raw',
    'Left_Heel_R_raw', 'Left_Heel_L_raw', 
    'Right_Heel_L_raw', 'Right_Heel_R_raw',

    "acceleration_Pelvis_x_local","acceleration_Pelvis_y_local","acceleration_Pelvis_z_local",
    "acceleration_RightForeArm_x_local","acceleration_RightForeArm_y_local","acceleration_RightForeArm_z_local",
    "acceleration_RightUpperLeg_x_local","acceleration_RightUpperLeg_y_local","acceleration_RightUpperLeg_z_local",
    "acceleration_RightLowerLeg_x_local","acceleration_RightLowerLeg_y_local","acceleration_RightLowerLeg_z_local",
    "acceleration_RightFoot_x_local","acceleration_RightFoot_y_local","acceleration_RightFoot_z_local",
    "acceleration_RightToe_x_local","acceleration_RightToe_y_local","acceleration_RightToe_z_local",
    "acceleration_LeftUpperLeg_x_local","acceleration_LeftUpperLeg_y_local","acceleration_LeftUpperLeg_z_local",
    "acceleration_LeftLowerLeg_x_local","acceleration_LeftLowerLeg_y_local","acceleration_LeftLowerLeg_z_local",
    "acceleration_LeftFoot_x_local","acceleration_LeftFoot_y_local","acceleration_LeftFoot_z_local",
    "acceleration_LeftToe_x_local","acceleration_LeftToe_y_local","acceleration_LeftToe_z_local",

    "angularVelocity_Pelvis_x_local","angularVelocity_Pelvis_y_local","angularVelocity_Pelvis_z_local",
    "angularVelocity_RightForeArm_x_local","angularVelocity_RightForeArm_y_local","angularVelocity_RightForeArm_z_local",
    "angularVelocity_RightUpperLeg_x_local","angularVelocity_RightUpperLeg_y_local","angularVelocity_RightUpperLeg_z_local",
    "angularVelocity_RightLowerLeg_x_local","angularVelocity_RightLowerLeg_y_local","angularVelocity_RightLowerLeg_z_local",
    "angularVelocity_RightFoot_x_local","angularVelocity_RightFoot_y_local","angularVelocity_RightFoot_z_local",
    "angularVelocity_RightToe_x_local","angularVelocity_RightToe_y_local","angularVelocity_RightToe_z_local",
    "angularVelocity_LeftUpperLeg_x_local","angularVelocity_LeftUpperLeg_y_local","angularVelocity_LeftUpperLeg_z_local",
    "angularVelocity_LeftLowerLeg_x_local","angularVelocity_LeftLowerLeg_y_local","angularVelocity_LeftLowerLeg_z_local",
    "angularVelocity_LeftFoot_x_local","angularVelocity_LeftFoot_y_local","angularVelocity_LeftFoot_z_local",
    "angularVelocity_LeftToe_x_local","angularVelocity_LeftToe_y_local","angularVelocity_LeftToe_z_local",

    'participant_id',  'walk_mode', 'stepcount'
    # 'participant_id',  'walk_mode'
]

In [ ]:
# This method uses a sliding window of 2 seconds with a step of 1 second (excluding the segments that have more than one class)
def calculate_statistical_features(folder_path):
    print("Working on", folder_path)
    df = pd.read_csv(os.path.join(folder_path, "merged.csv"))
    
    df = df[column_names]
    if 'stepcount' in df.columns:
        df = df.drop(columns=['stepcount'])
    # Invariant columns in the dataset (not features)
    additional_columns = ['participant_id',  'walk_mode']
    
    count_any_na = df.isna().any(axis=1).sum()
    if count_any_na > 0:
        print(folder_path, "contains a NaN")
        df.fillna(0, inplace=True)

    final_features_df = pd.DataFrame()
    
    # Window and step sizes based on the refresh rate of 60Hz
    window_size = 2 * 60
    step_size = 1 * 60
    
    unique_classes = df["walk_mode"].unique()

    for class_label in unique_classes:
        class_df = df[df["walk_mode"] == class_label].reset_index(drop=True)
        num_rows = len(class_df)

        for start in range(0, num_rows - window_size + 1, step_size):
            window_df = class_df.iloc[start:start + window_size]

            # Check if the window contains only one unique class label
            if window_df["walk_mode"].nunique() > 1:
                continue
            window_features = {}

            try:
                for feature in df.columns:  
                    if feature not in additional_columns: # Skip non-feature columns
                        feature_values = window_df[feature]

                        # Statistical features
                        window_features[feature + "_mean"] = feature_values.mean()
                        window_features[feature + "_min"] = feature_values.min()
                        window_features[feature + "_max"] = feature_values.max()
                        window_features[feature + "_std"] = feature_values.std()
                        window_features[feature + "_iqr"] = np.percentile(feature_values, 75) - np.percentile(feature_values, 25)
                        window_features[feature + "_mad"] = np.median(np.abs(feature_values - np.median(feature_values)))
                        window_features[feature + "_auc"] = integrate.simpson(feature_values)

                        # Entropy (Histogram-based method)
                        hist, bin_edges = np.histogram(feature_values, bins=10, density=True)
                        hist = hist[hist > 0]  # Remove zero entries to avoid log(0) in entropy
                        window_features[feature + "_entropy"] = entropy(hist)

                        # Skewness and Kurtosis
                        window_features[feature + "_skewness"] = skew(feature_values)
                        window_features[feature + "_kurtosis"] = kurtosis(feature_values)

                        # Frequency domain features
                        # Calculate the FFT
                        fft_values = np.fft.fft(feature_values)
                        dft_magnitude = np.abs(fft_values)

                        # Store the first five DFT coefficients
                        window_features[feature + "_dft1"] = dft_magnitude[1]
                        window_features[feature + "_dft2"] = dft_magnitude[2]
                        window_features[feature + "_dft3"] = dft_magnitude[3]
                        window_features[feature + "_dft4"] = dft_magnitude[4]
                        window_features[feature + "_dft5"] = dft_magnitude[5]

                        # Weighted Mean Frequency Calculation
                        frequencies = np.fft.fftfreq(len(feature_values))
                        weighted_mean_frequency = np.sum(frequencies[1:6] * dft_magnitude[1:6]) / np.sum(dft_magnitude[1:6])
                        window_features[feature + "_weighted_mean_frequency"] = weighted_mean_frequency

                # Add additional columns to the window features
                for col in additional_columns:
                    window_features[col] = window_df[col].iloc[0]  # Assuming all values are the same in the window
                window_features["walk_mode"] = class_label
                window_features["window_start"] = start
                window_features["window_end"] = start + window_size

                final_features_df = pd.concat([final_features_df, pd.DataFrame([window_features])], ignore_index=True)

            except Exception as e:
                print(f"Error processing class {class_label} at window {start}-{start + window_size}: {e}")
    final_features_df.to_csv(os.path.join(folder_path, "stat_freq_features(sliding window).csv"), index=False)

In [ ]:
# This method calculates statistical and frequency-based features for each batch having the same cycle
# It also excludes batches having different classes
def calculate_statistical_features(folder_path):
    print("Working on", folder_path)
    df = pd.read_csv(os.path.join(folder_path, "merged_gait_count_annotations.csv"))
    
    df = df[column_names]
    
    # Invariant columns in the dataset (not features)
    additional_columns = ['participant_id',  'walk_mode', 'stepcount']
    
    count_any_na = df.isna().any(axis=1).sum()
    if count_any_na > 0:
        print(folder_path, "contains a NaN")
        df.fillna(0, inplace=True)

    final_features_df = pd.DataFrame()

    # Group by 'stepcount' and process each group
    for stepcount, group in df.groupby('stepcount'):
        unique_walk_modes = group['walk_mode'].unique()

        if len(unique_walk_modes) > 1:
            continue

        current_class = unique_walk_modes[0]
        segment_features = {}

        # Calculate statistical features for each column except 'additional_columns'
        try:
            for feature in df.columns:
                if feature not in additional_columns:  # Skip non-feature columns
                    feature_values = group[feature]
                    
                    # Statistical features
                    segment_features[feature + "_mean"] = feature_values.mean()
                    segment_features[feature + "_min"] = feature_values.min()
                    segment_features[feature + "_max"] = feature_values.max()
                    segment_features[feature + "_std"] = feature_values.std()
                    segment_features[feature + "_iqr"] = np.percentile(feature_values, 75) - np.percentile(feature_values, 25)
                    segment_features[feature + "_mad"] = np.median(np.abs(feature_values - np.median(feature_values)))
                    segment_features[feature + "_auc"] = integrate.simpson(feature_values)

                    # Entropy (Histogram-based method)
                    hist, bin_edges = np.histogram(feature_values, bins=10, density=True)
                    hist = hist[hist > 0]  # Remove zero entries to avoid log(0) in entropy
                    segment_features[feature + "_entropy"] = entropy(hist)

                    # Skewness and Kurtosis
                    segment_features[feature + "_skewness"] = skew(feature_values)
                    segment_features[feature + "_kurtosis"] = kurtosis(feature_values)

                    # Frequency domain features
                    # Calculate the FFT
                    fft_values = np.fft.fft(feature_values)
                    dft_magnitude = np.abs(fft_values)

                    # Store the first five DFT coefficients
                    segment_features[feature + "_dft1"] = dft_magnitude[1]
                    segment_features[feature + "_dft2"] = dft_magnitude[2]
                    segment_features[feature + "_dft3"] = dft_magnitude[3]
                    segment_features[feature + "_dft4"] = dft_magnitude[4]
                    segment_features[feature + "_dft5"] = dft_magnitude[5]

                    # Weighted Mean Frequency Calculation
                    frequencies = np.fft.fftfreq(len(feature_values))
                    weighted_mean_frequency = np.sum(frequencies[1:6] * dft_magnitude[1:6]) / np.sum(dft_magnitude[1:6])
                    segment_features[feature + "_weighted_mean_frequency"] = weighted_mean_frequency

            # Add the additional columns (constant per group)
            for col in additional_columns:
                segment_features[col] = group[col].iloc[0]  # Assuming same value for the whole group

            segment_features["walk_mode"] = current_class

            final_features_df = pd.concat([final_features_df, pd.DataFrame([segment_features])], ignore_index=True)

        except Exception as e:
            print(f"Error processing batch with stepcount {stepcount}: {e}")
    final_features_df.to_csv(os.path.join(folder_path, "stat_freq_features(gait cycles).csv"), index=False)

In [ ]:
dataset_path = "data_set"

print("Calculating statistical features...")
for course_folder in os.listdir(dataset_path):
    course_folder_path = os.path.join(dataset_path, course_folder)
    if os.path.isdir(course_folder_path):
        for subfolder in os.listdir(course_folder_path):
            subfolder_path = os.path.join(course_folder_path, subfolder)
            if os.path.isdir(subfolder_path):
                calculate_statistical_features(subfolder_path)
print("Statistical features calculated successfully")

In [ ]:
base_dir = 'data_set'

dfs = []
for root, dirs, files in os.walk(base_dir):
    # if 'stat_freq_features(sliding window).csv' in files:
    if 'stat_freq_features(gait cycles).csv' in files:
        # csv_file_path = os.path.join(root, 'stat_freq_features(sliding window).csv')
        csv_file_path = os.path.join(root, 'stat_freq_features(gait cycles).csv')
        print("Merging", csv_file_path)
        
        df = pd.read_csv(csv_file_path)
        dfs.append(df)

# Concatenate all the DataFrames vertically
combined_df = pd.concat(dfs, ignore_index=True)
# combined_df.to_csv(os.path.join(base_dir, 'combined_stat_freq_features(sliding window).csv'), index=False)
combined_df.to_csv(os.path.join(base_dir, 'combined_stat_freq_features(gait cycles).csv'), index=False)
print("Merging complete")